# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

# 2. Import Data

In [2]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('../Data/ml-100k/u.data', sep='\t', names=column_names)

In [3]:
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


# 3. Preprocess Data

## 1. Membuat matrix User-Movie

### 1. Periksa jumlah user dan movie


In [4]:
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()

print(f"\nJumlah User: {n_users}")
print(f"Jumlah Movie: {n_items}")


Jumlah User: 943
Jumlah Movie: 1682


### 2. Buat matriks interaksi


In [5]:
interaction_matrix = df.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

In [6]:
interaction_matrix

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
940,0.0,0.0,0.0,2.0,0.0,0.0,4.0,5.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
941,5.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 3. Periksa baris dan kolom matriks interaksi

In [7]:
interaction_matrix.shape

(943, 1682)

## 2. Normalisasi value
    Normalisasi nilai rating ke rentang 0-1

In [8]:
matrix_values = interaction_matrix.values
min_rating = 0 # Kita anggap 0 adalah 'kosong'
max_rating = 5
normalized_matrix = matrix_values / max_rating

In [9]:
normalized_matrix

array([[1. , 0.6, 0.8, ..., 0. , 0. , 0. ],
       [0.8, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [1. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1. , 0. , ..., 0. , 0. , 0. ]], shape=(943, 1682))

## 3. Train Test Split (Hold Out)

### 1. Copy Data

#### 1. Train Data

In [10]:
train_data = normalized_matrix.copy()

In [11]:
train_data

array([[1. , 0.6, 0.8, ..., 0. , 0. , 0. ],
       [0.8, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [1. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1. , 0. , ..., 0. , 0. , 0. ]], shape=(943, 1682))

#### 2. Test Data

In [12]:
test_data = np.zeros(normalized_matrix.shape)

In [13]:
test_data

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(943, 1682))

### 2. Cari koordinat data rating

In [14]:
rows, cols = np.nonzero(normalized_matrix)

In [15]:
rows

array([  0,   0,   0, ..., 942, 942, 942], shape=(100000,))

In [16]:
cols

array([   0,    1,    2, ..., 1187, 1227, 1329], shape=(100000,))

### 3. Tentukan jumlah data testing

In [17]:
num_ratings = len(rows)
num_test = int(num_ratings * 0.2)

In [18]:
num_ratings

100000

In [19]:
num_test

20000

### 4. Acak indeks

In [20]:
np.random.seed(42)
idx = np.random.permutation(num_ratings)

In [21]:
idx

array([75721, 80184, 19864, ..., 76820,   860, 15795],
      shape=(100000,), dtype=int32)

### 5. Ambil indeks untuk data test

In [22]:
test_idx = idx[:num_test]

In [23]:
test_idx

array([75721, 80184, 19864, ..., 37862, 53421, 42410],
      shape=(20000,), dtype=int32)

### 6. Ambil baris dan kolom untuk data test

In [24]:
test_rows = rows[test_idx]
test_cols = cols[test_idx]

In [25]:
test_rows

array([692, 746, 200, ..., 343, 471, 387], shape=(20000,))

In [26]:
test_cols

array([381, 110, 211, ..., 683, 714,   8], shape=(20000,))

### 7. Input data asli ke data test

In [27]:
test_data[test_rows, test_cols] = normalized_matrix[test_rows, test_cols]

In [28]:
test_data

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(943, 1682))

### 8. Hapus data test dari data training

In [29]:
train_data[test_rows, test_cols] = 0

In [30]:
train_data

array([[1. , 0.6, 0.8, ..., 0. , 0. , 0. ],
       [0.8, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       ...,
       [1. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1. , 0. , ..., 0. , 0. , 0. ]], shape=(943, 1682))

# 4. Save data

In [31]:
# Simpan data training (80% dari total rating, dipakai untuk melatih model final)
np.save('..\\Data\\TrainTest\\train_data', train_data)

# Simpan data testing (20% dari total rating, dipakai untuk evaluasi model final)
np.save('..\\Data\\TrainTest\\test_data', test_data)

# Simpan matriks rating penuh sebelum di-split (dipakai CrossValidation.ipynb)
# CrossValidation membutuhkan data ini agar setiap fold bisa membuat split-nya sendiri
np.save('..\\Data\\TrainTest\\normalized_matrix', normalized_matrix)